# HER2 mBC Feasibility Analysis
**ConcertAI Technical Assessment — Azam Uddin**

This notebook is the **narrative layer** of the pipeline. All analysis logic lives in
`src/her2rwd/` (tested, config-driven Python modules). This notebook imports those modules,
runs the pipeline, and displays the outputs as a readable report.

> **All data is synthetic.** This analysis uses 150 synthetic patients generated for
> assessment purposes only. No real patient data is used.

**To reproduce:** `uv run jupyter nbconvert --to html --execute notebooks/analysis.ipynb --output reports/analysis_notebook.html`

---

## Section 0 — Setup & Input Integrity

Verify that `data/raw/` files are byte-identical to the originals in `../external/` using
SHA-256 hashes. This guarantees all downstream analysis runs on unmodified source data.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

# Resolve project root regardless of where Jupyter launches from
_here = Path().resolve()
_root = _here.parent if _here.name == "notebooks" else _here
os.chdir(_root)
sys.path.insert(0, str(_root / "src"))

from her2rwd.data_loader import check_input_integrity, load_raw
from her2rwd.data_quality import run_dq_checks

with open("config/analysis_rules.yaml") as f:
    cfg = yaml.safe_load(f)

hashes = check_input_integrity()
print(f"Input integrity: {len(hashes)} files verified OK")
for fname, h in hashes.items():
    print(f"  {fname}: {h[:20]}...")

Input integrity: 6 files verified OK
  data_dictionary.xlsx: 5474bcd4347ce8c01709...
  patient.csv: 13abfd38bdf8c66c69b4...
  diagnosis.csv: 0339f21d5196236a2511...
  staging.csv: c59ae51713763c5a328e...
  biomarker.csv: 6b10fe792d4e822b2b1a...
  medication.csv: 999b7a41383203e3c1ac...


## Section 1 — Data Profile & Data Quality

**19 automated checks** ran across all 5 raw tables. Every anomaly is logged — records are
never silently dropped. The table below shows what was found, and the decisions panel
immediately after explains exactly how each issue is handled and why.

In [2]:
con = load_raw()

print("Raw table row counts:")
for tbl in ['patient', 'diagnosis', 'staging', 'biomarker', 'medication']:
    n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f"  {tbl}: {n:,}")

Raw table row counts:
  patient: 150
  diagnosis: 333
  staging: 200
  biomarker: 1,059
  medication: 341


In [3]:
dq_summary = run_dq_checks(con, cfg)

errors   = int(dq_summary[dq_summary.severity == "error"]["n_flagged"].sum())
warnings = int(dq_summary[dq_summary.severity == "warning"]["n_flagged"].sum())
print(f"19 checks | {errors} errors | {warnings} warnings")
print()

# Show only checks that actually flagged something
flagged = dq_summary[dq_summary.n_flagged > 0][
    ["check_name", "table", "n_flagged", "severity", "action", "description"]
].reset_index(drop=True)
flagged

19 checks | 25 errors | 153 warnings



,check_name,table,n_flagged,severity,action,description
0,death_before_birth,patient,1,error,flag,date_of_death is earlier than date_of_birth
1,future_diagnosis_date,diagnosis,1,error,flag,diagnosis_date is after today
2,future_biomarker_date,biomarker,15,error,flag,biomarker report_date is after today
3,future_medication_date,medication,5,error,flag,med_date is after today
4,future_stage_date,staging,1,error,flag,stage_date is after today
5,post_death_diagnosis,diagnosis,16,warning,flag,diagnosis_date more than 30 days after date_of...
6,post_death_biomarker,biomarker,15,warning,flag,report_date more than 30 days after date_of_death
7,post_death_medication,medication,14,warning,flag,med_date more than 30 days after date_of_death
8,null_her2_variant,biomarker,55,warning,classify_as_unknown,HER2 record has NULL biomarker_variant — class...
9,unclassifiable_her2_variant,biomarker,6,warning,classify_as_unknown,HER2 variant is Pending or IHC equivocal — cla...


### Data Issues Found & Decisions Made

Every anomaly above was reviewed before any analysis ran. The table below pulls live counts
from the DQ checks and documents what was decided and why.

In [4]:
dq_log = pd.read_csv('outputs/dq/dq_log.csv')

def _n(check_name):
    """Count from dq_summary for a single check."""
    rows = dq_summary[dq_summary.check_name == check_name]
    return int(rows.iloc[0].n_flagged) if len(rows) else 0

def _n_union(check_names):
    """Unique patient count across multiple checks (avoids double-counting)."""
    return int(dq_log[dq_log.check_name.isin(check_names)]['patient_id'].nunique())

future_checks = ['future_diagnosis_date', 'future_biomarker_date',
                 'future_medication_date', 'future_stage_date']
post_death_checks = ['post_death_diagnosis', 'post_death_biomarker', 'post_death_medication']

# Non-C50 diagnosis codes — not a DQ check (comorbidities are expected), run inline
n_comorbidity_patients = con.execute("""
    SELECT COUNT(DISTINCT patient_id) FROM diagnosis
    WHERE diagnosis_code NOT LIKE 'C50%'
""").fetchone()[0]

decisions_data = [
    ("Death before birth",
     _n("death_before_birth"),
     "Exclude from cohort",
     "Impossible record — causes errors in age/survival calculation"),
    ("Future-dated records (any table)",
     _n_union(future_checks),
     "Flag + exclude downstream",
     "Future dates are data entry errors; cannot be clinical events"),
    ("Post-death events > 30-day grace",
     _n_union(post_death_checks),
     "Flag + exclude",
     "Events >30 days after death are implausible; 30-day grace retained for admin lag"),
    ("Orphan biomarker IDs (not in patient table)",
     _n("orphan_biomarker"),
     "Exclude record",
     "Cannot link to demographics or index date"),
    ("NULL HER2 variant",
     _n("null_her2_variant"),
     "Classify as unknown",
     "No IHC score → cannot assign HER2 category"),
    ("HER2 variant = Pending or IHC equivocal",
     _n("unclassifiable_her2_variant"),
     "Classify as unknown",
     "Indeterminate result — no category assignable"),
    ("HER2 result='Negative' but variant='IHC 1+'",
     _n("her2_result_variant_conflict"),
     "Variant wins → HER2-low",
     "'Negative' = old binary label; IHC 1+ = HER2-low per ASCO/CAP 2023"),
    ("ER/PR NULL variant (no % available)",
     _n("er_pr_null_variant_fallback"),
     "Fall back to result text",
     "ASCO/CAP allows result text when % absent; ER-low cannot be assessed"),
    ("Duplicate biomarker same date",
     _n("duplicate_biomarker_same_date"),
     "Keep curated (flag=1); else more specific variant",
     "Curation flag signals abstractor review"),
    ("Non-breast metastatic ICD code",
     _n("non_breast_metastatic_code"),
     "Exclude from index date",
     "mBC index date requires C50.x confirmation"),
    ("NULL stage_group",
     _n("null_stage_group"),
     "Fall back to m_stage = M1",
     "M-stage available in 100% of staging records; used as metastatic signal"),
    ("Non-C50 codes in diagnosis table (comorbidities)",
     n_comorbidity_patients,
     "Ignore for eligibility; retain for context",
     "Non-C50 codes are comorbidities (diabetes, COPD etc.), not breast cancer events"),
]

decisions_df = pd.DataFrame(
    decisions_data,
    columns=["Issue found", "n patients", "Decision", "Why"]
)
decisions_df

,Issue found,n patients,Decision,Why
0,Death before birth,1,Exclude from cohort,Impossible record — causes errors in age/survi...
1,Future-dated records (any table),2,Flag + exclude downstream,Future dates are data entry errors; cannot be ...
2,Post-death events > 30-day grace,18,Flag + exclude,Events >30 days after death are implausible; 3...
3,Orphan biomarker IDs (not in patient table),2,Exclude record,Cannot link to demographics or index date
4,NULL HER2 variant,55,Classify as unknown,No IHC score → cannot assign HER2 category
5,HER2 variant = Pending or IHC equivocal,6,Classify as unknown,Indeterminate result — no category assignable
6,HER2 result='Negative' but variant='IHC 1+',40,Variant wins → HER2-low,'Negative' = old binary label; IHC 1+ = HER2-l...
7,ER/PR NULL variant (no % available),89,Fall back to result text,ASCO/CAP allows result text when % absent; ER-...
8,Duplicate biomarker same date,3,Keep curated (flag=1); else more specific variant,Curation flag signals abstractor review
9,Non-breast metastatic ICD code,2,Exclude from index date,mBC index date requires C50.x confirmation


## Section 2 — Cohort Construction

Attrition table showing how 150 patients are filtered to the analysis cohort.
Index date definition (earliest metastatic event), de novo vs. recurrent split.
Sources: `outputs/tables/cohort.csv`, `outputs/tables/attrition.csv`.
Populated in **Phase 02 (cohort-index-date)**.

In [5]:
# Phase 02 will populate this section
# from her2rwd.cohort import build_cohort
# cohort, attrition = build_cohort(con, cfg)
# display(attrition)

## Section 3 — HER2 Classification

Distribution of HER2 variants found in the data, window selection logic, and tie-break outcomes.
Final per-patient HER2 categories (positive / low / ultralow / null / unknown) with audit columns.
Source: `outputs/tables/patient_her2.csv`.
Populated in **Phase 03 (biomarker-derivation)**.

In [6]:
# Phase 03 will populate this section
# from her2rwd.her2 import derive_her2
# patient_her2 = derive_her2(con, cohort, cfg)
# display(patient_her2.head())

## Section 4 — HR Status

ER/PR percentage distribution, HR-positive and ER-low classification.
Source: `outputs/tables/patient_hr.csv`.
Populated in **Phase 03 (biomarker-derivation)**.

In [7]:
# Phase 03 will populate this section
# from her2rwd.hormone_receptor import derive_hr
# patient_hr = derive_hr(con, cohort, cfg)
# display(patient_hr.head())

## Section 5 — Q1: HER2 Distribution

**Research question:** What is the HER2 distribution in this mBC dataset, overall and across
four strata (HR status × line of therapy)? How does it compare to published literature?

Includes Wilson 95% CIs, small-cell flags (n < 11), and sensitivity analysis.
Source: `outputs/tables/q1_*.csv`.
Populated in **Phase 04 (q1-distribution)**.

In [8]:
# Phase 04 will populate this section
# from her2rwd.q1_distribution import compute_q1
# q1_overall, q1_strata = compute_q1(patient_her2, patient_hr, cohort, cfg)
# display(q1_overall)

## Section 6 — Q2: Treatment Lines

Regimen grouping and line-of-therapy approximation for the cohort.
First-line and second-line distribution by HER2 category.
Source: `outputs/tables/patient_lines.csv`.
Populated in **Phase 05 (treatment-lines)**.

In [9]:
# Phase 05 will populate this section
# from her2rwd.treatment import build_treatment_lines
# patient_lines = build_treatment_lines(con, cohort, cfg)
# display(patient_lines.head())

## Section 7 — Q2: Enhertu Uptake

**Research question:** What is the real-world uptake of Enhertu (trastuzumab deruxtecan) by
subindication (HER2-positive 2L+, HER2-low 2L+, HER2-ultralow)? What is the time to first
Enhertu dose? Who is receiving it outside label (discordance)?

Sources: `outputs/tables/q2_*.csv`.
Populated in **Phase 06 (q2-uptake)**.

In [10]:
# Phase 06 will populate this section
# from her2rwd.q2_uptake import compute_q2
# q2_results = compute_q2(patient_her2, patient_hr, patient_lines, cohort, cfg)
# display(q2_results['subindication_summary'])

## Section 8 — Fit-for-Purpose Verdict

Summary assessment of what this synthetic asset can and cannot support:
- Which claims can be made with confidence?
- Which findings are directional only (n < 11, data gaps)?
- What gaps would need to be addressed before a commercial study?

Populated in **Phase 06 (q2-uptake)**.

In [11]:
# Phase 06 will populate this section
# Verdict table: claim | confidence | limiting factor | mitigation

---

## Appendix A — Data Quality Log

Full per-patient DQ flags from `outputs/dq/dq_log.csv`.

In [12]:
print(f"Total flagged records: {len(dq_log)}")
dq_log

Total flagged records: 346


,patient_id,check_name,severity,action,description
0,PAT_001,death_before_birth,error,flag,date_of_death is earlier than date_of_birth
1,PAT_042,future_diagnosis_date,error,flag,diagnosis_date is after today
2,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
3,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
4,PAT_042,future_biomarker_date,error,flag,biomarker report_date is after today
...,...,...,...,...,...
341,PAT_052,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
342,PAT_015,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
343,PAT_077,duplicate_biomarker_same_date,warning,apply_curation_tiebreak,Same patient_id + report_date + biomarker_name...
344,PAT_134,non_breast_metastatic_code,warning,exclude_from_index_date,diagnosis_type='metastatic' with non-C50 ICD c...


## Appendix B — Analytical Assumptions Register

Every rule that governs this analysis, with its approved value, the clinical rationale,
and the ADR that documents the full decision context. All values are sourced from
`config/analysis_rules.yaml` — nothing is hard-coded.

In [13]:
assumptions = [
    # ── Eligibility ────────────────────────────────────────────────────────
    ("Eligibility", "Breast cancer diagnosis required",
     "require_breast_code = true", "ADR-002",
     "Only C50.x ICD codes count. 135 comorbidity records in data are ignored for eligibility."),
    ("Eligibility", "Minimum age at metastatic index date",
     "min_age_at_index_years = 18", "ADR-002",
     "ASCO standard for adult oncology studies."),
    ("Eligibility", "Valid date of birth required",
     "valid_dob_required = true", "ADR-002",
     "Patients with death before birth (1 found) are excluded."),
    # ── DQ Handling ───────────────────────────────────────────────────────
    ("DQ Handling", "Future-dated records",
     "future_date_action = flag", "ADR-003",
     "Flag and exclude downstream. Never silently drop — keeps audit trail."),
    ("DQ Handling", "Post-death event grace period",
     "post_death_grace_days = 30", "ADR-003",
     "Events within 30 days of death retained (admin/lab lag). Beyond 30 days excluded."),
    ("DQ Handling", "Curation flag priority",
     "curation_priority = curated_first", "ADR-003",
     "Curated records (flag=1) preferred over uncurated. Fallback to uncurated if no curated exists."),
    # ── HER2 Classification ───────────────────────────────────────────────
    ("HER2 Classification", "Biomarker look-up window",
     "pre_index = 365d, post_index = 90d", "ADR-007",
     "Captures primary workup biomarkers (73 patients). Excludes tests >12 months before metastatic staging."),
    ("HER2 Classification", "Variant beats result text",
     "conflict_resolution = variant_beats_result_text", "ADR-008",
     "IHC score is objective. 'Negative' label is old binary paradigm. 40 patients reclassified IHC 1+ → HER2-low."),
    ("HER2 Classification", "Tie-break order (multiple records in window)",
     "curation_flag → variant_specificity → closest_to_index", "ADR-008",
     "Curated + specific + recent = most reliable single record per patient."),
    ("HER2 Classification", "Unclassifiable variants → unknown",
     "Pending, IHC equivocal, NULL → unknown", "ADR-006",
     "No IHC score available; cannot assign HER2-low/positive/null category."),
    # ── HR Status ─────────────────────────────────────────────────────────
    ("HR Status", "HR-positive threshold",
     "positive_threshold_pct = 1%", "ADR-009",
     "ASCO/CAP 2020: ≥1% nuclear ER or PR staining = HR-positive."),
    ("HR Status", "ER-low flag",
     "er_low_max_pct = 10%", "ADR-009",
     "ER 1–10% = ER-low subgroup. Distinct biology relevant for endocrine + CDK4/6 inhibitor decisions."),
    ("HR Status", "NULL variant fallback",
     "null_variant_fallback = result_text", "ADR-009",
     "89 patients have no ER/PR %. Fall back to Positive/Negative label. ASCO/CAP allows this. ER-low cannot be assessed."),
    ("HR Status", "Numeric % beats result text",
     "pct_beats_result_text = true", "ADR-009",
     "Lab % is objective measurement. Text label may reflect local cutoffs differing from ASCO/CAP 1%."),
]

register = pd.DataFrame(assumptions, columns=[
    "Category", "Decision", "Rule / Value", "ADR", "Rationale"
])
register.set_index(["Category", "Decision"], inplace=True)
register

Rule / Value  \
Category            Decision                                                                                          
Eligibility         Breast cancer diagnosis required                                     require_breast_code = true   
                    Minimum age at metastatic index date                                min_age_at_index_years = 18   
                    Valid date of birth required                                          valid_dob_required = true   
DQ Handling         Future-dated records                                                  future_date_action = flag   
                    Post-death event grace period                                        post_death_grace_days = 30   
                    Curation flag priority                                        curation_priority = curated_first   
HER2 Classification Biomarker look-up window                                     pre_index = 365d, post_index = 90d   
                    Variant beats result text                       conflict_resolution = variant_beats_result_text   
                    Tie-break order (multiple records in window)  curation_flag → variant_specificity → closest_...   
                    Unclassifiable variants → unknown                        Pending, IHC equivocal, NULL → unknown   
HR Status           HR-positive threshold                                               positive_threshold_pct = 1%   
                    ER-low flag                                                                er_low_max_pct = 10%   
                    NULL variant fallback                                       null_variant_fallback = result_text   
                    Numeric % beats result text                                        pct_beats_result_text = true   

                                                                      ADR  \
Category            Decision                                                
Eligibility         Breast cancer diagnosis required              ADR-002   
                    Minimum age at metastatic index date          ADR-002   
                    Valid date of birth required                  ADR-002   
DQ Handling         Future-dated records                          ADR-003   
                    Post-death event grace period                 ADR-003   
                    Curation flag priority                        ADR-003   
HER2 Classification Biomarker look-up window                      ADR-007   
                    Variant beats result text                     ADR-008   
                    Tie-break order (multiple records in window)  ADR-008   
                    Unclassifiable variants → unknown             ADR-006   
HR Status           HR-positive threshold                         ADR-009   
                    ER-low flag                                   ADR-009   
                    NULL variant fallback                         ADR-009   
                    Numeric % beats result text                   ADR-009   

                                                                                                          Rationale  
Category            Decision                                                                                         
Eligibility         Breast cancer diagnosis required              Only C50.x ICD codes count. 135 comorbidity re...  
                    Minimum age at metastatic index date                  ASCO standard for adult oncology studies.  
                    Valid date of birth required                  Patients with death before birth (1 found) are...  
DQ Handling         Future-dated records                          Flag and exclude downstream. Never silently dr...  
                    Post-death event grace period                 Events within 30 days of death retained (admin...  
                    Curation flag priority                        Curated records (flag=1) preferred over uncura...  
HER2 Classification B

## Appendix C — Validation Report

Invariant checks verifying internal consistency of all pipeline outputs.
Source: `audit/validation_report.md`.
Populated in **Phase 07 (deliverables)**.

In [14]:
# Phase 07 will populate this section
# from her2rwd.validation import run_validation
# validation_results = run_validation()
# display(validation_results)